In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
!pip install transformers==4.40.0 \
             datasets==2.16.0 \
             accelerate==0.33.0 \
             evaluate==0.4.1 \
             jiwer==3.0.3 \
             librosa==0.10.1 \
             soundfile==0.12.1 \
             numpy==1.26.4 \
             peft==0.10.0 \
             -q --force-reinstall

In [ ]:
import torch
import librosa
import soundfile
import evaluate
import transformers
import datasets
import accelerate
import jiwer

print(f"torch version        : {torch.__version__}")
print(f"transformers version : {transformers.__version__}")
print(f"datasets version     : {datasets.__version__}")
print(f"evaluate version     : {evaluate.__version__}")
print(f"GPU available        : {torch.cuda.is_available()}")
print(f"GPU name             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM total (GB)      : {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
from huggingface_hub import login
login(token="hf_token", add_to_git_credential=False)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google/fleurs", "hi_in", trust_remote_code=True)
print(dataset)

In [ ]:
print(f"Train samples      : {len(dataset['train'])}")
print(f"Validation samples : {len(dataset['validation'])}")
print(f"Test samples       : {len(dataset['test'])}")

In [ ]:
for i in range(3):
    sample = dataset['train'][i]
    print(f"Sample {i+1}")
    print(f"Transcript : {sample['transcription']}")
    print(f"Audio SR   : {sample['audio']['sampling_rate']}")
    print(f"Audio len  : {len(sample['audio']['array'])} samples")
    print(f"Duration   : {round(len(sample['audio']['array']) / sample['audio']['sampling_rate'], 2)} seconds")
    print()

In [ ]:
empty_train = [i for i, s in enumerate(dataset['train']['transcription']) if s.strip() == ""]
print(f"Empty transcripts in train : {len(empty_train)}")

empty_test = [i for i, s in enumerate(dataset['test']['transcription']) if s.strip() == ""]
print(f"Empty transcripts in test  : {len(empty_test)}")

In [ ]:
cols_to_remove = ['id', 'num_samples', 'path', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']

dataset = dataset.remove_columns(cols_to_remove)
print(dataset)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on : {device}")
print(f"Model parameters: {model.num_parameters() / 1e6:.1f}M")

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on : {device}")
print(f"Model parameters: {model.num_parameters() / 1e6:.1f}M")

In [ ]:
from tqdm import tqdm
import torch

model.eval()

predictions = []
references = []


test_subset = dataset["test"].select(range(200))

for sample in tqdm(test_subset, desc="Running baseline inference"):
    
    input_features = processor(
        sample["audio"]["array"],
        sampling_rate=sample["audio"]["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(device)

    
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="hi",
            task="transcribe"
        )

    
    pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    ref_text = sample["transcription"]

    predictions.append(pred_text)
    references.append(ref_text)

print(f"\nDone. Total samples evaluated: {len(predictions)}")

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

baseline_wer = wer_metric.compute(predictions=predictions, references=references)
baseline_cer = cer_metric.compute(predictions=predictions, references=references)

print(f"Baseline WER : {baseline_wer * 100:.2f}%")
print(f"Baseline CER : {baseline_cer * 100:.2f}%")

In [ ]:
print("Sample predictions vs references:\n")
for i in range(5):
    print(f"Reference  : {references[i]}")
    print(f"Prediction : {predictions[i]}")
    print()

In [ ]:
baseline_results = {
    "wer": round(baseline_wer * 100, 2),
    "cer": round(baseline_cer * 100, 2)
}
print("Baseline results saved:", baseline_results)

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="Hindi",
    task="transcribe"
)

print("Processor loaded successfully")

In [ ]:
def prepare_dataset(batch):
    
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]

    
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

In [ ]:
dataset = dataset.map(
    prepare_dataset,
    remove_columns=["audio", "transcription"],
    num_proc=1
)

print(dataset)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready")

In [ ]:
sample_batch = [dataset["train"][i] for i in range(4)]
batch = data_collator(sample_batch)

print(f"input_features shape : {batch['input_features'].shape}")
print(f"labels shape         : {batch['labels'].shape}")
print(f"labels sample        : {batch['labels'][0][:10]}")

In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("Metrics function ready")

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper-small-hindi",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    eval_steps=200,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

print("Training arguments configured")

In [ ]:
from transformers import WhisperForConditionalGeneration, GenerationConfig

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

model.generation_config = GenerationConfig.from_pretrained("openai/whisper-small")
model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print("Model ready for training")

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print("Trainer initialized")

In [ ]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_pretrained("openai/whisper-small")
model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/kaggle/working/whisper-small-hindi-final")
processor.save_pretrained("/kaggle/working/whisper-small-hindi-final")

print("Model saved to /kaggle/working/whisper-small-hindi-final")

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import os


model = WhisperForConditionalGeneration.from_pretrained("/kaggle/working/whisper-small-hindi/checkpoint-800")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")


model.save_pretrained("/kaggle/working/whisper-small-hindi-final")
processor.save_pretrained("/kaggle/working/whisper-small-hindi-final")

print("Model saved to /kaggle/working/whisper-small-hindi-final")
print("Files:", os.listdir("/kaggle/working/whisper-small-hindi-final"))

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch

model = WhisperForConditionalGeneration.from_pretrained("/kaggle/working/whisper-small-hindi-final")
processor = WhisperProcessor.from_pretrained("/kaggle/working/whisper-small-hindi-final")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Fine-tuned model loaded on: {device}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google/fleurs", "hi_in", trust_remote_code=True)
print(f"Test samples: {len(dataset['test'])}")

In [ ]:
from tqdm import tqdm

finetuned_predictions = []
finetuned_references = []

test_subset = dataset["test"].select(range(200))

for sample in tqdm(test_subset, desc="Running fine-tuned inference"):
    input_features = processor(
        sample["audio"]["array"],
        sampling_rate=sample["audio"]["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="hi",
            task="transcribe"
        )

    pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    ref_text = sample["transcription"]

    finetuned_predictions.append(pred_text)
    finetuned_references.append(ref_text)

print(f"Done. Total samples: {len(finetuned_predictions)}")

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

finetuned_wer = wer_metric.compute(predictions=finetuned_predictions, references=finetuned_references)
finetuned_cer = cer_metric.compute(predictions=finetuned_predictions, references=finetuned_references)

print(f"Fine-tuned WER : {finetuned_wer * 100:.2f}%")
print(f"Fine-tuned CER : {finetuned_cer * 100:.2f}%")

In [ ]:
print("=" * 50)
print(f"{'Model':<25} {'WER':>10} {'CER':>10}")
print("=" * 50)
print(f"{'whisper-small (base)':<25} {'86.46%':>10} {'52.87%':>10}")
print(f"{'whisper-small (fine-tuned)':<25} {f'{finetuned_wer*100:.2f}%':>10} {f'{finetuned_cer*100:.2f}%':>10}")
print("=" * 50)
print(f"{'WER Improvement':<25} {f'{86.46 - finetuned_wer*100:.2f}%':>10}")
print(f"{'CER Improvement':<25} {f'{52.87 - finetuned_cer*100:.2f}%':>10}")

In [ ]:
print("Sample predictions vs references:\n")
for i in range(10):
    print(f"Reference  : {finetuned_references[i]}")
    print(f"Prediction : {finetuned_predictions[i]}")
    print()

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch
import librosa
import numpy as np


device = "cuda" if torch.cuda.is_available() else "cpu"


finetuned_model = WhisperForConditionalGeneration.from_pretrained("/kaggle/working/whisper-small-hindi-final").to(device)
finetuned_processor = WhisperProcessor.from_pretrained("/kaggle/working/whisper-small-hindi-final")


base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
base_processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")

print("Both models loaded")

def transcribe(model, processor, audio_array, sampling_rate):
    input_features = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to(device)
    
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="hi",
            task="transcribe"
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google/fleurs", "hi_in", trust_remote_code=True)


demo_indices = [5, 15, 25]

for idx in demo_indices:
    sample = dataset["test"][idx]
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    duration = round(len(audio) / sr, 1)
    
    base_pred = transcribe(base_model, base_processor, audio, sr)
    ft_pred = transcribe(finetuned_model, finetuned_processor, audio, sr)
    
    print(f" Sample {idx} ({duration}s) ")
    print(f"Reference   : {sample['transcription']}")
    print(f"Base model  : {base_pred}")
    print(f"Fine-tuned  : {ft_pred}")
    print()